# Handoff and Customer Success: Transfer the Capability, Not the Folder

> **Riverside at 2 AM:** the policy index is stale, an alert has fired, and the on-call editor has five handoff files but no clear first action. The FDE is unavailable. Can the customer team contain the problem, preserve evidence, and decide whether to roll back?
>
> The frozen case has reached the broad-availability gate. The repository contains source assets and procedures, but no retained cloud evidence, committed SLO, rehearsed rollback, or accepted post-hypercare quality owner. This notebook turns those gaps into visible owners and blockers.
>
> Artifact labels such as `HOF-*`, `OPS-*`, `RBK-*`, `DRL-*`, and `CHG-*` keep links stable. Evidence labels tell reviewers whether a statement was observed, modeled, or accepted by an authorized Riverside role.

| Part | Decision to earn | Output |
|---:|---|---|
| 0 | Why does the polished document dump fail? | Failed package assessment |
| 1 | What makes a package traceable? | Package index |
| 2 | Is the service ready for ownership? | Readiness review |
| 3 | Which signals produce which actions? | Dashboard and alert guide |
| 4 | Can operators contain and recover safely? | Runbook set |
| 5 | How may policy behavior and thresholds change? | Change process |
| 6 | Who responds, when, and with what authority? | Support matrix |
| 7 | Can named operators perform the work? | Training and drills |
| 8 | What blocks acceptance and FDE exit? | Sign-off and exit gates |
| 9 | How does ownership persist after hypercare? | Health review and backlog |
| 10 | What can Riverside now operate, and what still blocks exit? | Capability recap and exit checklist |

## 0 · The Challenge

> **It is 2 AM.** Riverside's policy index is six hours stale. The on-call editor opens the handoff folder and finds a dashboard export, a generic runbook, a support spreadsheet, training slides, and a signed acceptance form. None says who owns this alert or whether rollback is safe.

### Before: files without capability

- The alert does not name an accountable Riverside role.
- The runbook does not identify the first safe action or evidence to preserve.
- The support sheet does not say whether 2 AM is covered or who may authorize re-enablement.
- The signed form does not prove that an operator has practiced the response.

### After: an ownership path

The repaired package links `stale index -> ROLE-SUPPORT-L2 -> freeze index changes -> preserve trace and version IDs -> RBK-RIV-INDEX -> escalate to the Editorial Director -> revalidate before re-enable`.

### Spot the failure

Riverside received every filename in the checklist. Which review outcome is correct?

1. **Presence pass** - artifact presence proves completeness.
2. **FDE-dependent pass** - accept because the FDE can remain available for questions.
3. **Capability failure** - reject because no capability is linked to an owner, action, evidence, drill, limitation, or revalidation trigger.

Choose the outcome and list the first three questions the on-call editor still cannot answer. Then run the validator to compare your diagnosis with the required capability fields.

In [ ]:
# ── Fail the Document-Dump Handoff ───────────────────────────────────────────
REQUIRED_CAPABILITY_FIELDS = {
    "artifact_id", "capability", "receiving_owner", "decision_or_action",
    "evidence_ref", "drill_status", "limitations", "revalidate_on",
}

def assess_handoff(records):
    issues = []
    for position, record in enumerate(records, start=1):
        missing = sorted(REQUIRED_CAPABILITY_FIELDS - set(record))
        if missing:
            issues.append({"record": position, "missing": missing})
    return issues

document_dump = [
    {"file": "dashboard.pdf"},
    {"file": "runbook.docx"},
    {"file": "support.xlsx"},
    {"file": "training-slides.pptx"},
    {"file": "acceptance.pdf"},
]
document_dump_issues = assess_handoff(document_dump)
actual_outcome = "Capability failure" if document_dump_issues else "Presence pass"
print(f"FAIL: {len(document_dump_issues)} of {len(document_dump)} records lack capability-transfer fields.")
for issue in document_dump_issues:
    print(f"  record {issue['record']}: missing {', '.join(issue['missing'])}")
print(f"Prediction closed: {actual_outcome} - filenames cannot prove ownership or safe action.")
print("  Takeaway: file presence is not operating evidence.")

#### Why this design choice matters

The validator ignores filenames because Riverside must operate during an incident, not admire a folder. Consider a PageTurn timeout: the request may have committed even though the response was lost. The operator must know who can pause writes, who can inspect committed state, and who can approve re-enablement. Calling the FDE is not a recovery design.

```mermaid
flowchart TD
    A["2 AM alert or PageTurn timeout"] --> B{"Named Riverside owner?"}
    B -- No --> X["Handoff remains open"]
    B -- Yes --> C["Take first safe action"]
    C --> D["Preserve trace, release, policy, and target-state evidence"]
    D --> E{"Outcome and authority clear?"}
    E -- No --> F["Escalate through the support matrix"]
    E -- Yes --> G["Run rollback, reconciliation, or approved degraded path"]
    F --> H["Authorized re-enable decision"]
    G --> H
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style X fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style H fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

**Checkpoint:** all five document-dump records fail because none links the artifact to an owner, action, evidence record, drill, limitation, or revalidation trigger.

A richer package can still fail if it copies stale support facts or leaves the FDE as the owner. The repair must point back to the frozen Riverside records and name the customer role that accepts the work.

---
## Part 1 - Build a Traceable Package

Load the frozen case instead of copying and silently editing its support facts. Then repair the package while preserving the difference between structural completeness and operational evidence.

**Common Pitfalls**

| Wrong | Right | Why it matters |
|---|---|---|
| Copy support facts into a private handoff sheet | Reference frozen IDs and retain source/version | Copies drift without a visible decision |
| Assign the FDE as receiving owner | Name the customer role with authority and escalation | Availability after handoff cannot depend on private context |
| Mark a drill `pass` because a runbook exists | Keep `not_run` until retained drill evidence exists | Authored procedure is not measured operator capability |

**Quick Health Check:** every package row needs artifact ID, capability, receiving owner, action, evidence reference, drill status, limitation, and revalidation trigger.

In [ ]:
# ── Load Frozen Facts and Repair Package Links ────────────────────────────────
from pathlib import Path
import json

def find_repo_root(start):
    for candidate in (start, *start.parents):
        if (candidate / "AUTHORING_GUIDE.md").is_file() and (candidate / "learning" / "role-based-tracks" / "fde" / "shared").is_dir():
            return candidate
    raise FileNotFoundError("Run from inside the ai-portfolio checkout.")

repo_root = find_repo_root(Path.cwd().resolve())
fixture_path = repo_root / "learning/role-based-tracks/fde/shared/fixtures/riverside-engagement-v1.json"
engagement = json.loads(fixture_path.read_text(encoding="utf-8"))
support = engagement["support_and_handoff"]
open_unknowns = {item["unknown_id"]: item for item in engagement["unknowns"]}

repaired_package = [
    {
        "artifact_id": "OPS-DASH-01",
        "capability": "interpret health and choose an operating boundary",
        "receiving_owner": "ROLE-SUPPORT-L2",
        "decision_or_action": "continue, contain, stop ramp, or investigate",
        "evidence_ref": "dashboard and telemetry drill: not run",
        "drill_status": "not_run",
        "limitations": "live Azure signals and alert delivery are unvalidated",
        "revalidate_on": "signal, sampling, retention, release, or owner change",
    },
    {
        "artifact_id": "HOF-02",
        "capability": "contain, roll back, compensate, and re-enable",
        "receiving_owner": "ROLE-SUPPORT-L2",
        "decision_or_action": "execute the scenario runbook or escalate",
        "evidence_ref": "timed recovery drill: not run",
        "drill_status": "not_run",
        "limitations": "procedures are authored source, not rehearsed recovery evidence",
        "revalidate_on": "dependency, release, index, policy, runbook, or owner change",
    },
]
repaired_issues = assess_handoff(repaired_package)
print("PASS: required package fields are present." if not repaired_issues else repaired_issues)
print("BLOCKED: capability drills are still not run.")
print("  Takeaway: structural completeness and operational acceptance are separate gates.")

**Your turn:** Change one `receiving_owner` to `ROLE-FDE`. Predict that the exit rule will fail, then use the next cell to compare the actual owner scan with your prediction. Restore the customer owner afterward.

The ownership fields now exist, but Riverside still has no live cloud evidence or completed drills. That makes readiness a decision over evidence, not document quality.

---
## Part 2 - Riverside Wants to Launch. Is It Ready?

The autumn deadline is approaching. Leadership asks for a launch answer, but the platform still lacks live deployment evidence, committed SLOs, rollback rehearsal, approved residency, and reconciled production profiles. Each missing gate needs an owner, a due date, and a safe consequence.

| Gate | Owner | Needed by | Evidence needed | If missing |
|---|---|---|---|---|
| Live service path | IT owner | Before shadow | Retained deployment and smoke evidence for the named release/region | Keep traffic off the service |
| Capacity and SLO | Service owner + Finance | Before canary | Scoped latency, volume, quota, support, and cost evidence | Keep the target proposed and narrow the cohort |
| Rollback and PageTurn recovery | Release owner | Before write-enabled canary | Timed rollback plus committed-state reconciliation drill | Keep writes disabled |
| Regional processing | Security/compliance owner | Before UK/EU exposure | Approved service-by-service residency review and deployed configuration | Hold regional data exposure |
| After-hours support | Commercial/support owner | Before broad availability | Accepted hours, severity path, exclusions, and escalation contacts | Do not promise uncovered support |
| Post-hypercare quality | Editorial Director | Before FDE exit | Accepted recurring owner, review schedule, and trigger | Keep handoff open |

**What happens when a gate is skipped?** If Riverside enables PageTurn writes before the recovery drill, a timeout may leave a committed status change behind. Traffic rollback restores the old release but does not repair the title state. The launch decision must therefore remain `BLOCKED` until the owner supplies the named proof.

**Your turn:** choose one gate, write its owner and due date, and state the exact exposure that remains disabled while evidence is missing. Then run the next cell and compare your decision with the five authored blockers.

In [ ]:
# ── Check Ownership and Apply an Evidence-Aware Readiness Gate ────────────────
# CHANGE THIS: set one receiving_owner to "ROLE-FDE" and predict the result.
fde_owned = [record["artifact_id"] for record in repaired_package
             if record["receiving_owner"] == "ROLE-FDE"]
readiness_rows = [
    {"gate": "live_azure_path", "state": "source_only", "critical": True},
    {"gate": "slo_capacity_cost", "state": "unset", "critical": True},
    {"gate": "rollback_rehearsal", "state": "not_run", "critical": True},
    {"gate": "support_after_hours", "state": open_unknowns["UNK-RIV-006"]["status"], "critical": True},
    {"gate": "post_hypercare_quality_owner", "state": open_unknowns["UNK-RIV-010"]["status"], "critical": True},
]
PASS_STATES = {"measured_pass", "customer_accepted"}
blockers = [row for row in readiness_rows if row["critical"] and row["state"] not in PASS_STATES]
print(f"FAIL: FDE remains receiving owner for {fde_owned}." if fde_owned
      else "PASS: no package capability is assigned to the FDE as receiving owner.")
print(f"BLOCKED: {len(blockers)} critical readiness gates lack accepted evidence.")
for blocker in blockers:
    print(f"  {blocker['gate']}: {blocker['state']}")
print("  Takeaway: source presence and planned procedures cannot pass an operational gate.")

---
## Part 3 - An Alert Fires. Then What?

A dashboard is useful only when the on-call person can turn a signal into a safe decision. Generic uptime cannot tell Riverside whether policy retrieval is current, access is denied correctly, PageTurn committed twice, or telemetry itself has stopped reporting.

| Signal | Owner | First safe action | Runbook | Escalation |
|---|---|---|---|---|
| Deadline success drops | L2 Operations | Stop ramp or isolate the failing boundary | Service degradation runbook | Incident commander if deadline exposure continues |
| Current-policy retrieval falls | Editorial Director | Freeze index changes or disable guidance | Index freshness runbook | Data owner and Rights if current policy cannot be proved |
| Forbidden access appears | Security | Fail closed and retain all approved evidence | `RBK-RIV-IDENTITY` | Security incident authority before any re-enable |
| PageTurn outcome is ambiguous | L2 Operations | Pause writes and query committed state | `RBK-RIV-TOOL-AMBIGUOUS` | Workflow owner for reconciliation or compensation |
| Telemetry becomes stale | L2 Operations | Stop evidence-dependent changes | Telemetry blind-spot runbook | Service owner if visibility is not restored |

**Exercise:** `ALT-RIV-001` detects forbidden access. If retention changes from `retain_all` to `random_1_percent`, the event becomes cheaper to store but may disappear from the evidence trail. Decide whether that is acceptable before running the safety rule below.

In a future authorized session, change only that sampling value, inspect the reported failure, and restore `retain_all`. Then add the missing clear condition: who decides that negative tests are sufficient to re-enable traffic?

**Ready check:** each alert names its scope, freshness, owner, first action, runbook, escalation authority, evidence retention, blind spot, and clear condition. The PageTurn timeout also distinguishes stopping exposure from repairing committed state.

In [ ]:
# ── Link Signals and Alerts to Decisions ─────────────────────────────────────
dashboard_signals = [
    {"signal": "deadline_success", "decision": "stop ramp or isolate boundary", "owner": "ROLE-SUPPORT-L2", "blind_spot": "does not prove answer quality"},
    {"signal": "current_policy_retrieval", "decision": "freeze index or disable guidance", "owner": "ROLE-EDITORIAL-DIRECTOR", "blind_spot": "small slices may hide rare failures"},
    {"signal": "forbidden_access", "decision": "fail closed and engage Security", "owner": "ROLE-SECURITY", "blind_spot": "rare events must not be sampled away"},
    {"signal": "workflow_reconciliation", "decision": "pause writes and query committed state", "owner": "ROLE-SUPPORT-L2", "blind_spot": "timeout does not reveal commit state"},
    {"signal": "telemetry_freshness", "decision": "stop evidence-dependent changes", "owner": "ROLE-SUPPORT-L2", "blind_spot": "missing telemetry does not prove health"},
]
alerts = [
    {"alert_id": "ALT-RIV-001", "severity": "SEV-1", "owner": "ROLE-SECURITY", "first_action": "fail closed", "runbook": "RBK-RIV-IDENTITY", "sampling": "retain_all"},
    {"alert_id": "ALT-RIV-003", "severity": "SEV-2", "owner": "ROLE-SUPPORT-L2", "first_action": "pause writes and query state", "runbook": "RBK-RIV-TOOL-AMBIGUOUS", "sampling": "priority"},
]
signal_issues = [row["signal"] for row in dashboard_signals
                 if not all(row.get(field) for field in ("decision", "owner", "blind_spot"))]
alert_issues = []
for alert in alerts:
    missing = [field for field in ("owner", "first_action", "runbook") if not alert.get(field)]
    if alert["severity"] == "SEV-1" and alert["sampling"] != "retain_all":
        missing.append("retain_all safety evidence")
    if missing:
        alert_issues.append((alert["alert_id"], missing))
actual_sampling_outcome = "fail because a cross-tenant access signal may be sampled away" if alert_issues else "remain safe with retain_all"
print("PASS: signals map to decisions and alerts retain safety fields."
      if not signal_issues and not alert_issues else (signal_issues, alert_issues))
print(f"Prediction closed: alerts {actual_sampling_outcome}.")
print("  Takeaway: charts and notifications become controls only when they enable safe action.")

---
## Part 4 - Runbooks Separate Containment, Rollback, and Compensation

Riverside's PageTurn incident is the trap: the tool commits a workflow update, loses the response, and receives a retry. Rolling back model traffic does not undo the committed status change.

```mermaid
flowchart TD
    T["Trigger"] --> C["Contain and preserve evidence"]
    C --> K{"Outcome known?"}
    K -- No --> Q["Query target state"]
    Q --> A{"State now known?"}
    A -- No --> E["Escalate with evidence"]
    A -- Yes --> D{"Recovery type"}
    K -- Yes --> D
    D --> R["Rollback exposure"]
    D --> P["Compensate committed action"]
    D --> F["Approved degraded mode"]
    R --> V["Revalidate and obtain approval"]
    P --> V
    F --> V
    style T fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style K fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style Q fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style A fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style R fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style P fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style V fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

**Decision check:** A PageTurn write timed out after the remote system committed it. Does routing traffic to the previous model version restore the workflow, does a new retry key make the action safe, or must the team query the original business key and reconcile the committed state? The next cell maps the seeded tool incident to the bounded first action.

**Common Pitfalls**

| Wrong | Right | Why it matters |
|---|---|---|
| Retry every timeout | Query target state with the stable key | The far side may already have committed |
| Roll back traffic and declare recovery | Reconcile committed actions separately | Traffic changes cannot undo side effects |
| Use `compensation` for any cleanup | Reserve compensation for an explicit semantic counter-action; use correction or reconciliation when that is the real operation | Recovery language determines authority and audit evidence |
| Delete suspect deployments or logs | Preserve evidence and known-good state | Investigation and rollback need retained state |
| Re-enable after a health check | Run positive and negative gates with authority | Liveness is not safe behavior |

**Quick Health Check:** identify the failure boundary, stop new exposure, preserve the original business key, determine commit state, select rollback/correction/compensation precisely, retain the action history, and require re-enablement authority.

**Why this leads to the next step:** A correct runbook can still be invalidated by an unreviewed policy edit or an unfunded support promise. Recovery therefore feeds directly into controlled change and explicit service boundaries.

In [ ]:
# ── Map Seeded Incidents to Bounded First Actions ─────────────────────────────
runbook_map = {
    "policy": ("RBK-RIV-STALE-RETRIEVAL", "disable affected guidance and freeze index promotion"),
    "data": ("RBK-RIV-DATA-SYNC", "quarantine version and preserve lineage"),
    "identity": ("RBK-RIV-IDENTITY", "disable affected route and preserve access evidence"),
    "model": ("RBK-RIV-ROLLBACK", "require abstention and stop candidate exposure"),
    "tool": ("RBK-RIV-TOOL-AMBIGUOUS", "pause writes and query committed state"),
    "infrastructure": ("RBK-RIV-PROVIDER", "fail closed or use approved degraded mode"),
}
for incident in engagement["seeded_incidents"]:
    runbook_id, first_action = runbook_map[incident["domain"]]
    print(f"{incident['incident_id']} -> {runbook_id}: {first_action}")
tool_first_action = runbook_map["tool"][1]
print(f"Prediction closed: query and reconcile - '{tool_first_action}'.")
print("  Takeaway: the failure boundary selects the runbook; one restart guide cannot.")

---
## Parts 5 and 6 - Govern Change and Bound Support

Policy and threshold edits change authority or exposure. Treat them as versioned releases with positive and negative evaluation, approval, canary, rollback, and retained evidence. Support targets also need staffed hours, communication authority, vendor paths, and explicit exclusions.

```mermaid
flowchart LR
    R["Evidence-backed request"] --> I["Authority and impact"]
    I --> V["Versioned policy and tests"]
    V --> E["Positive and negative evaluation"]
    E --> A{"Authorized approval?"}
    A -- No --> H["Hold"]
    A -- Yes --> C["Bounded canary"]
    C --> G{"Observed gates pass?"}
    G -- No --> B["Rollback and preserve evidence"]
    G -- Yes --> P["Promote and monitor"]
    style R fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style I fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style V fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style A fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style H fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style P fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

Riverside covers 08:00-18:00 Europe/London on weekdays, but `UNK-RIV-006` leaves out-of-hours coverage unresolved. `UNK-RIV-010` leaves post-hypercare model/retrieval quality ownership unresolved. Neither can be silently assigned to the FDE.

**Decision check:** A stakeholder asks to lower a quality threshold because the canary is behind schedule. Will the change pass as an operational tweak, pass because rollback exists, or remain blocked until an authorized approver accepts versioned positive/negative evidence and bounded exposure? The next cell reports the failed gate by name.

**Your turn:** Write the evidence request, approval authority, exposure limit, automatic response, and retained decision that would make the proposal reviewable. Do not change the value until the gate exists.

**Common Pitfalls**

| Wrong | Right | Why it matters |
|---|---|---|
| Lower a gate to keep the calendar | Reopen the evidence and authority decision | Calendar pressure is not validation |
| Publish response targets without staffed coverage | Bind targets to hours, roles, vendors, and exclusions | An unfunded promise becomes incident debt |
| Leave post-hypercare quality with the FDE | Name a receiving owner and review cadence | Temporary delivery support becomes a hidden dependency |

**Quick Health Check:** policy changes carry artifact version, positive/negative cases, authority, canary, rollback target, retained decision, and revalidation trigger; support promises carry staffed hours, severities, communication authority, vendor paths, exclusions, and expiry.

**Why this leads to the next step:** Versioned controls and bounded support still prove nothing about operator behavior. The next gate is a drill in which a person must preserve safety and authority under pressure.

In [ ]:
# ── Gate a Policy Change and Expose Support Blockers ─────────────────────────
policy_change = {
    "change_id": "CHG-POL-RIV-001", "trigger": "INC-RIV-001",
    "versioned_artifact": True, "positive_cases": True, "negative_cases": True,
    "authorized_approval": False, "bounded_canary": True, "known_good_rollback": True,
}
required_change_gates = (
    "versioned_artifact", "positive_cases", "negative_cases",
    "authorized_approval", "bounded_canary", "known_good_rollback",
)
failed_change_gates = [gate for gate in required_change_gates if not policy_change[gate]]
support_checks = {
    "covered_hours_defined": bool(support["covered_hours"]),
    "severity_targets_defined": bool(support["severity_definitions"]),
    "after_hours_decided": open_unknowns["UNK-RIV-006"]["status"] != "open",
    "post_hypercare_quality_owner_decided": open_unknowns["UNK-RIV-010"]["status"] != "open",
}
support_blockers = [name for name, passed in support_checks.items() if not passed]
print(f"BLOCKED policy change: missing {failed_change_gates}." if failed_change_gates
      else "PASS: policy change may enter its approved cohort.")
print(f"BLOCKED support acceptance: missing {support_blockers}." if support_blockers
      else "PASS: support boundary and handback owner are decided.")
print("Prediction closed: remain blocked until authorized approval and bounded evidence exist.")
print("  Takeaway: source cannot approve itself, and the FDE cannot become unfunded permanent support.")

---
## Parts 7 and 8 - Drills Gate Acceptance and Exit

Training is complete only when Riverside operators can perform the work without private FDE guidance. Use this progressive drill ladder; each step adds a harder decision while retaining the evidence from the previous one.

| Step | Riverside scenario | Pass | Fail |
|---:|---|---|---|
| 1. Trace | Follow one failed request across gateway, retrieval/model, policy, tool, and audit records | Operator identifies the failed boundary using safe IDs and preserves required evidence | Operator uses raw customer content, loses correlation, or cannot name the boundary |
| 2. Rollback | A candidate regresses after a PageTurn action may have committed | Operator stops exposure, restores the named target, and reconciles the committed action separately | Operator claims traffic rollback undoes the PageTurn write |
| 3. Isolation | A cross-tenant retrieval attempt is detected | Operator fails closed, retains evidence, engages Security, and waits for authorized re-enable | Operator continues traffic, samples evidence, or self-approves re-enable |
| 4. Threshold change | Editorial requests a lower policy threshold before a deadline | Operator uses a versioned proposal, representative slices, approval, canary, rollback, and retained decision | Operator edits production directly or copies an example threshold without evidence |

Access-revocation and deletion propagation remain required alongside the ladder. A signature cannot override a failed critical drill, missing owner, unsupported claim, or unresolved support boundary.

**Skipped training failure:** if the first real isolation incident occurs before the team drills, the operator may restore traffic without Security approval. Fast recovery and a manager signature cannot rescue that critical authority failure; acceptance remains blocked.

**Exercise:** in a future authorized session, mark `DRL-RIV-ISOLATION` as `pass` and set `critical_failure=True`. Confirm that the next cell still blocks acceptance, then restore the authored `not_run` state.

Every drill record needs the scenario, release/environment, operator, expected and observed actions, timing, critical errors, evidence handling, reviewer, remediation, and rerun result. Even passing evidence expires when data, policy, models, thresholds, staffing, or ownership changes.

In [ ]:
# ── Keep Training, Acceptance, and FDE Exit Evidence-Gated ───────────────────
drills = [
    {"drill_id": "DRL-RIV-TRACE", "status": "not_run", "critical_failure": False},
    {"drill_id": "DRL-RIV-ROLLBACK", "status": "not_run", "critical_failure": False},
    {"drill_id": "DRL-RIV-DELETE", "status": "not_run", "critical_failure": False},
    {"drill_id": "DRL-RIV-ISOLATION", "status": "not_run", "critical_failure": False},
    {"drill_id": "DRL-RIV-THRESHOLD", "status": "not_run", "critical_failure": False},
]
training_blockers = [drill["drill_id"] for drill in drills
                     if drill["status"] != "pass" or drill["critical_failure"]]
acceptance_inputs = {
    "readiness_blockers": len(blockers),
    "training_blockers": len(training_blockers),
    "support_blockers": len(support_blockers),
    "fde_owned_capabilities": len(fde_owned),
    "fde_only_access_removed": False,
    "authorized_signoff": False,
}
open_acceptance = {
    name: value for name, value in acceptance_inputs.items()
    if (isinstance(value, bool) and not value) or (isinstance(value, int) and value > 0)
}
print(f"BLOCKED training: {training_blockers}")
print(f"REJECT acceptance/FDE exit: {open_acceptance}" if open_acceptance
      else "PASS: bounded acceptance and FDE exit gates are satisfied.")
print("Prediction closed: critical failures remain non-compensating; timing and signature cannot rescue them.")
print("  Takeaway: elapsed time and signatures do not substitute for demonstrated ownership.")

---
## Part 9 - Recurring Health and the Evidence-Based Backlog

Handoff decays when workflow, data, policy, model, support, or ownership changes. Review customer value, retrieval/generation quality, identity/policy, reliability, capacity/cost, changes, training, and retirement together.

```mermaid
flowchart LR
    H["Health review"] --> E["Evidence, incidents, feedback, expiry"]
    E --> D{"Decision"}
    D --> C["Continue"]
    D --> X["Change or investigate"]
    D --> R["Reduce exposure or retire"]
    X --> B["Evidence-based backlog"]
    B --> G["Validation and rollout gate"]
    G --> H
    style H fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style X fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style R fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

**Review checks:** preserve claim classes; show expired evidence; reopen data, identity, evaluation, rollout, or handoff gates on material change; keep retirement available.

**Your turn:** Assume workflow value stays below the customer-approved floor while support burden exceeds the accepted envelope for three review periods. Write the `[Measured]` evidence, `[Modeled]` forward cost, authorized decision owner, and stop/revalidation conditions needed to choose among remediation, reduced exposure, deterministic replacement, and retirement.

**Common Pitfalls**

| Wrong | Right | Why it matters |
|---|---|---|
| Prioritize the loudest feature request | Rank by safety, authority, customer value, and evidence gap | Novelty can displace the actual operating risk |
| Carry expired evidence forward | Reopen the affected gate on material change | Acceptance has a scope and a shelf life |
| Treat retirement as failure | Compare ongoing value, risk, and support burden | Stopping can be the correct product decision |

**Quick Health Check:** every backlog item has trigger, affected criterion, claim class, owner, validation, exposure decision, stop condition, and next review; safety and authority blockers outrank feature demand.

**Why this closes the loop:** The recurring review shows what was structurally built, what remains unmeasured, and why Riverside must still withhold acceptance.

In [ ]:
# ── Keep Backlog Priority Tied to Evidence ────────────────────────────────────
backlog = [
    {"id": "BLG-RIV-001", "trigger": "UNK-RIV-010", "safety_or_authority": True, "owner": "PER-RIV-001", "validation": "accepted post-hypercare ownership record"},
    {"id": "BLG-RIV-004", "trigger": "platform limitation: no rollback rehearsal", "safety_or_authority": True, "owner": "ROLE-IT-OWNER", "validation": "timed staging rollback and re-enablement evidence"},
    {"id": "BLG-RIV-006", "trigger": "INC-RIV-001", "safety_or_authority": False, "owner": "PER-RIV-002", "validation": "versioned retrieval and policy evaluation"},
]
backlog_issues = [item["id"] for item in backlog
                  if not all(item.get(field) for field in ("trigger", "owner", "validation"))]
ordered_backlog = sorted(backlog, key=lambda item: (not item["safety_or_authority"], item["id"]))
print("PASS: each item has trigger, owner, and validation." if not backlog_issues else backlog_issues)
print("Priority order:", [item["id"] for item in ordered_backlog])
print("  Takeaway: authority and safety blockers cannot be outvoted by feature demand.")

## Part 10 - What Riverside Can Now Operate

```mermaid
flowchart LR
    A["Document dump rejected"] --> B["Capability links authored"]
    B --> C["Readiness blockers exposed"]
    C --> D["Signals mapped to action"]
    D --> E["Recovery language separated"]
    E --> F["Change, support, and drills gated"]
    F --> G["Recurring ownership defined"]
    G --> H["Acceptance remains BLOCKED"]
    style A fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style H fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

### Riverside capability recap

Riverside now has an operating model instead of a folder handoff. The package links each capability to an owner, alert, first action, evidence reference, drill, limitation, and revalidation trigger. Operators can distinguish containment from deployment rollback, then use reconciliation to determine whether PageTurn committed a business action before anyone proposes correction or compensation.

That is a usable design, not proof of readiness. Live Azure evidence, committed SLO/capacity/cost evidence, a retained rollback rehearsal, out-of-hours support, post-hypercare quality ownership, removal of private FDE access, and authorized Riverside sign-off are still missing. The current acceptance decision remains `BLOCKED`.

### What you can now do

| Riverside task | What this notebook gives you | What still counts as proof |
|---|---|---|
| Build a handoff package an operator can follow | Capability, owner, signal, action, runbook, limitation, and revalidation fields | Riverside must approve the named owners and scope `[Customer-validated]` |
| Decide whether an alert is actionable | Signal-to-owner-to-first-action mapping with safe defaults | Retained environment results and calibrated thresholds `[Measured]` |
| Respond to a PageTurn write failure | Separate containment, rollback, reconciliation, correction, and compensation paths | A retained drill showing operator action and committed-state recovery `[Measured]` |
| Set support and change boundaries | Hours, authority, escalation, exclusions, approvals, and expiry are explicit | Staffed commitments and approved service terms `[Customer-validated]`; vendor promises require external confirmation |
| Decide whether the FDE can exit | Non-compensating checks for evidence, drills, access transfer, ownership, and sign-off | All critical checks must pass; synthetic fixture results remain practice evidence only |

### Riverside exit decision checklist

- [ ] Live telemetry and alert delivery are retained as `[Measured]` evidence.
- [ ] Named Riverside operators complete the rollback and committed-action recovery drill.
- [ ] Support hours, out-of-hours ownership, and vendor escalation are accepted by authorized roles `[Customer-validated]`.
- [ ] SLO, capacity, and cost claims keep `[Modeled]` labels until live measurement replaces them.
- [ ] Private FDE access is removed or transferred and independently verified.
- [ ] Post-hypercare quality review has a Riverside owner, cadence, and revalidation triggers.
- [ ] Authorized approvers sign the scoped acceptance record with limitations and expiry.

### Keep these operating rules

1. Handoff transfers decisions and authority, not filenames.
2. Source completeness cannot be relabeled as measured operator capability.
3. A dashboard matters only when a scoped signal leads to a safe action and named owner.
4. Deployment rollback stops new exposure; it does not reverse a committed business action.
5. Use reconciliation, correction, and compensation precisely because each implies different evidence and authority.
6. Critical safety, authority, and evidence failures never average into acceptance.
7. FDE exit requires customer-owned access, drills, support, recurring review, and revalidation triggers.

> **Forward:** carry the unresolved Riverside blockers, capability evidence requirements, and `BLOCKED` acceptance decision into `09-capstone`, where the complete engagement package must make every dependency and claim traceable.